# Aetheris Pre-Operative Risk ML Model Training

This notebook trains a Random Forest Classifier on the `MImic IV Dataset.xlsx` to predict 30-day post-operative complications (`Complication 30d (0/1)`). We will encode categorical variables, scale numerical features, and export the trained model along with the preprocessor to the `aetheris-backend/app/ml/` directory.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

## 1. Load the Dataset
We load the Excel dataset, skipping the title row.

In [2]:
df = pd.read_excel('../MImic IV Dataset.xlsx', header=1)
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (1200, 26)


,Subject ID,Age (yrs),Gender,Weight (kg),Height (cm),BMI,Systolic BP,Diastolic BP,BP Category,Hypertension (0/1),...,Hemoglobin (g/dL),Anemia (0/1),Prior Surgery Count,Prior Surgeries,Med Count,Medication List,ASA Class,Risk Score (0-100),Risk Category,Complication 30d (0/1)
0,MIMIC-100000,66,M,94.0,166.9,33.7,131,76,Stage 1 HTN,0,...,14.3,0,3,Laparoscopic Procedure; Cholecystectomy; Appen...,2,Rosuvastatin; Apixaban,2,22.2,Moderate,0
1,MIMIC-100001,55,F,50.3,151.8,21.8,109,59,Normal,0,...,14.9,0,2,C-Section; Knee Replacement,7,Aspirin; Gabapentin; Metformin; Furosemide; Si...,2,26.7,Moderate,0
2,MIMIC-100002,69,F,60.7,163.9,22.6,143,92,Stage 2 HTN,1,...,12.1,0,1,Hip Replacement,4,Rosuvastatin; Apixaban; Metoprolol; Lisinopril,2,16.0,Low,0
3,MIMIC-100003,85,F,103.3,160.4,40.2,137,77,Stage 1 HTN,0,...,12.3,0,3,Hip Replacement; Hernia Repair; Cholecystectomy,5,Rosuvastatin; Insulin Glargine; Furosemide; Me...,3,35.1,Moderate,0
4,MIMIC-100004,53,F,63.6,168.0,22.5,94,73,Normal,0,...,12.3,0,0,NaN,4,Rosuvastatin; Metformin; Albuterol; Empagliflozin,3,18.1,Low,0


## 2. Preprocessing
Select relevant clinical features. We drop high-cardinality or target-leakage columns like IDs or computed Risk Scores/Categories.

In [3]:
# Define Target
TARGET = 'Complication 30d (0/1)'

# Define Features
NUMERICAL = ['Age (yrs)', 'Weight (kg)', 'Height (cm)', 'BMI', 
             'Systolic BP', 'Diastolic BP', 'Creatinine (mg/dL)', 
             'eGFR (mL/min)', 'Hemoglobin (g/dL)', 'Prior Surgery Count', 'Med Count']

CATEGORICAL = ['Gender', 'Diabetes (0/1)', 'Hypertension (0/1)', 'ASA Class']

X = df[NUMERICAL + CATEGORICAL].copy()
y = df[TARGET]

# Convert binary ints masquerading as strings if necessary
X['Diabetes (0/1)'] = X['Diabetes (0/1)'].astype(str)
X['Hypertension (0/1)'] = X['Hypertension (0/1)'].astype(str)

# Handle Missing Values inline
X[NUMERICAL] = X[NUMERICAL].fillna(X[NUMERICAL].median())
X[CATEGORICAL] = X[CATEGORICAL].fillna('Unknown')

## 3. Build Training Pipeline
We use a `ColumnTransformer` to scale numbers and one-hot encode categories.

In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERICAL),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)
    ])

# We use Random Forest to get probability scores for risk assessment
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))])

## 4. Train and Evaluate

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

Classification Report:
               precision    recall  f1-score   support

           0       0.92      1.00      0.96       221
           1       0.00      0.00      0.00        19

    accuracy                           0.92       240
   macro avg       0.46      0.50      0.48       240
weighted avg       0.85      0.92      0.88       240

ROC-AUC Score: 0.5395332221957608


C:\Users\rudra\Downloads\Aetheris-main\Aetheris-main\aetheris-backend\venv\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\rudra\Downloads\Aetheris-main\Aetheris-main\aetheris-backend\venv\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\rudra\Downloads\Aetheris-main\Aetheris-main\aetheris-backend\venv\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division`

## 5. Export Model for Backend
Save the pipeline to `aetheris-backend/app/ml/mimic_preop_model.pkl` so FastAPI can load it.

In [6]:
os.makedirs('../aetheris-backend/app/ml', exist_ok=True)
model_path = '../aetheris-backend/app/ml/mimic_preop_model.pkl'
joblib.dump(pipeline, model_path)
print(f"Model successfully exported to {model_path}")

Model successfully exported to ../aetheris-backend/app/ml/mimic_preop_model.pkl
